# Tratamento de batimentos cardíacos com PySpark

Pipeline local para os dados gerados por `gerador_batimentos_local.py`. Entrada: XLSX local. O XLSX é convertido para CSV apenas como etapa de ingestão, porque o Spark SQL não possui leitor XLSX nativo. O tratamento e as análises são executados pelo PySpark.

Objetivos: tipagem explícita, seleção/projeção antecipada, filtros antecipados, reutilização com cache, agregações e inspeção do plano físico/DAG/AQE.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, stddev, min, max, count, sum as spark_sum, round as spark_round, when
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, BooleanType
import pandas as pd
import os

spark = (SparkSession.builder
    .appName('AnaliseBatimentosPySpark')
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.sql.shuffle.partitions', '6')
    .getOrCreate())

print('Spark:', spark.version)
print('AQE:', spark.conf.get('spark.sql.adaptive.enabled'))
print('Shuffle partitions:', spark.conf.get('spark.sql.shuffle.partitions'))

## 1. Ingestão do XLSX

Spark não lê XLSX diretamente com `spark.read.csv/parquet`. Para manter o experimento local e simples, o pandas faz somente a leitura do arquivo Excel e cria um CSV intermediário. A partir daí, o processamento é Spark. Em um cenário maior, o ideal é converter a origem para Parquet/Delta antes do processamento.

In [ ]:
XLSX_PATH = '/home/ubuntu/dados_batimentos.xlsx'
CSV_PATH = '/home/ubuntu/dados_batimentos_intermediario.csv'

pd.read_excel(XLSX_PATH, sheet_name='Dados Batimentos').to_csv(CSV_PATH, index=False)

schema = StructType([
    StructField('messageId', IntegerType(), False),
    StructField('deviceId', StringType(), False),
    StructField('heartRate', DoubleType(), False),
    StructField('heartRateTarget', DoubleType(), False),
    StructField('activityState', IntegerType(), False),
    StructField('activityLabel', StringType(), False),
    StructField('bpmAlert', BooleanType(), False),
    StructField('timestamp', StringType(), False),
    StructField('deviceIndex', IntegerType(), False),
    StructField('timeSinceStart', StringType(), False)
])

df_raw = (spark.read.option('header', True).schema(schema).csv(CSV_PATH))
df_raw.printSchema()
df_raw.show(5, truncate=False)

## 2. Baseline: plano do pipeline original

Este bloco reproduz a ideia do pipeline original: filtro de anomalias e agregação. Use `explain('formatted')` para registrar os `Exchange` e comparar com o pipeline otimizado.

In [ ]:
df_anormal_baseline = df_raw.filter((col('heartRate') > 150) | (col('heartRate') < 40))
stats_baseline = df_raw.agg(avg('heartRate').alias('media_bpm'), stddev('heartRate').alias('desvio_bpm'))

print('PLANO BASELINE - filtro')
df_anormal_baseline.explain('formatted')
print('PLANO BASELINE - agregação')
stats_baseline.explain('formatted')

## 3. Tratamento otimizado

`select` reduz o conjunto de colunas usado no processamento; `filter` elimina registros inválidos cedo. O Catalyst pode aplicar Predicate Pushdown e Projection Pruning. Mesmo assim, escrever o pipeline de forma clara ajuda a expressar a intenção. O resultado tratado é persistido em memória para ser reutilizado por múltiplas análises sem reler/recalcular toda a linhagem.

In [ ]:
COLUNAS_ANALISE = ['messageId','deviceId','heartRate','heartRateTarget','activityState','activityLabel','bpmAlert','timestamp']

df_tratado = (df_raw
    .select(*COLUNAS_ANALISE)
    .filter(col('heartRate').isNotNull())
    .filter((col('heartRate') >= 30) & (col('heartRate') <= 220))
    .withColumn('timestamp', col('timestamp').cast('timestamp'))
    .withColumn('faixa_bpm',
        when(col('heartRate') < 50, 'Baixo')
        .when(col('heartRate') > 150, 'Alto')
        .otherwise('Normal'))
)

df_tratado.cache()
print('Registros tratados:', df_tratado.count())
df_tratado.show(10, truncate=False)

## 4. Anomalias e estatísticas

Estas operações reutilizam `df_tratado`. A agregação `groupBy` é uma dependência larga (wide), portanto normalmente cria `Exchange`/shuffle. O filtro é narrow e não precisa redistribuir os dados.

In [ ]:
df_anormal = df_tratado.filter((col('heartRate') > 150) | (col('heartRate') < 40))
print('Registros anormais:', df_anormal.count())
df_anormal.select('timestamp','deviceId','heartRate','activityLabel').show(20, truncate=False)

stats = df_tratado.agg(
    spark_round(avg('heartRate'), 2).alias('media_bpm'),
    spark_round(stddev('heartRate'), 2).alias('desvio_bpm'),
    min('heartRate').alias('min_bpm'),
    max('heartRate').alias('max_bpm'),
    count('*').alias('total_registros')
)
stats.show()

In [ ]:
resumo_dispositivo = (df_tratado
    .groupBy('deviceId')
    .agg(
        count('*').alias('total'),
        spark_round(avg('heartRate'), 2).alias('media_bpm'),
        min('heartRate').alias('min_bpm'),
        max('heartRate').alias('max_bpm'),
        spark_sum(when(col('bpmAlert') == True, 1).otherwise(0)).alias('alertas')
    )
    .orderBy('deviceId'))

resumo_dispositivo.explain('formatted')
resumo_dispositivo.show(truncate=False)

## 5. Inspeção do plano e codegen

O `.explain('formatted')` mostra o plano físico e permite localizar `Exchange`. `codegen` evidencia Whole-Stage Code Generation quando aplicável.

In [ ]:
print('=== PLANO FÍSICO DO RESUMO ===')
resumo_dispositivo.explain('formatted')
print('=== CODEGEN ===')
resumo_dispositivo.explain('codegen')

## 6. AQE e configurações para a evidência do relatório

AQE pode adaptar o plano durante a execução, inclusive coalescer partições pequenas e mudar a estratégia de join quando houver evidência de que um lado é pequeno. Aqui não há join real entre tabelas, então não devemos inventar um `BroadcastExchange`: o relatório deve registrar apenas os `Exchange` efetivamente observados.

In [ ]:
print('AQE enabled:', spark.conf.get('spark.sql.adaptive.enabled'))
print('Default shuffle partitions:', spark.conf.get('spark.sql.shuffle.partitions'))
print('Adaptive coalesce:', spark.conf.get('spark.sql.adaptive.coalescePartitions.enabled'))

## 7. Gravação local em Parquet

Parquet é preferível ao CSV para uma camada tratada porque é colunar, preserva tipos e permite que o Spark leia apenas as colunas necessárias. A gravação é uma ação e pode aparecer como um job separado na Spark UI.

In [ ]:
OUTPUT_PATH = '/home/ubuntu/processed_bpm_parquet'
df_tratado.write.mode('overwrite').parquet(OUTPUT_PATH)
print('Dados tratados salvos em:', OUTPUT_PATH)

df_tratado.unpersist()

## 8. Evidências para o relatório

Depois de executar os blocos, abra a Spark UI em `http://localhost:4040` no servidor Spark. Na EC2, o acesso remoto exige túnel SSH (ver guia entregue junto com este notebook). Na aba Jobs, abra o job principal e use `DAG Visualization`; em Stages, registre tempo, tasks, Shuffle Read/Write e o número de stages. Use essas métricas reais no PDF; não invente números.